# Performance Tuning for Agent Workloads
Companion notebook for the **Performance Tuning for Agent Workloads** section of Chapter 12.

Latency comes from a mix of model calls, tool waits, retries, queue buildup, and uneven request size -- tune the full path, not just one node.

This is the book's concurrency-guard pattern, unchanged -- limit fan-out per step so parallel tool calls don't overload a quota or a third-party API:

In [1]:
import asyncio

sem = asyncio.Semaphore(8)


async def guarded(call):
    async with sem:
        return await call()

In [2]:
async def tool_a():
    await asyncio.sleep(0.01)
    return "a-result"


async def tool_b():
    await asyncio.sleep(0.01)
    return "b-result"


async def tool_c():
    await asyncio.sleep(0.01)
    return "c-result"


results = await asyncio.gather(
    guarded(tool_a),
    guarded(tool_b),
    guarded(tool_c),
)
print(results)

['a-result', 'b-result', 'c-result']


**Batching**: good for embeddings, reranking, moderation -- cap both batch size *and* max wait time, so a slow trickle of requests doesn't stall behind a batch that never fills:

In [3]:
import time


def make_batcher(max_batch_size: int, max_wait_s: float):
    pending: list[str] = []
    batch_started_at: float | None = None

    def add(item: str) -> list[str] | None:
        nonlocal batch_started_at
        pending.append(item)
        if batch_started_at is None:
            batch_started_at = time.perf_counter()
        should_flush = len(pending) >= max_batch_size or (time.perf_counter() - batch_started_at) >= max_wait_s
        if should_flush:
            flushed, pending[:] = pending[:], []
            batch_started_at = None
            return flushed
        return None

    return add


add_to_batch = make_batcher(max_batch_size=4, max_wait_s=1.0)
for item in ["chunk-1", "chunk-2", "chunk-3", "chunk-4", "chunk-5"]:
    flushed = add_to_batch(item)
    if flushed:
        print("flushing batch:", flushed)

flushing batch: ['chunk-1', 'chunk-2', 'chunk-3', 'chunk-4']


**Backpressure**: cap in-flight runs and shed excess work instead of letting timeouts spread through the whole graph. This is the book's `limits` config, enforced:

In [4]:
limits = {
    "max_inflight_runs": 200,
    "queue_max": 1000,
    "shed_on_overload": True,
}


def admit_request(inflight_count: int, queue_depth: int, limits: dict) -> str:
    if inflight_count < limits["max_inflight_runs"]:
        return "run_now"
    if queue_depth < limits["queue_max"]:
        return "queue"
    if limits["shed_on_overload"]:
        return "reject"  # shed load instead of letting it pile up into a timeout storm
    raise RuntimeError("no shedding configured and queue is full")


print(admit_request(inflight_count=50, queue_depth=0, limits=limits))
print(admit_request(inflight_count=200, queue_depth=500, limits=limits))
print(admit_request(inflight_count=200, queue_depth=1000, limits=limits))

run_now
queue
reject


**Tail latency** (p95/p99) usually matters more than the average -- set timeouts on slow tools, keep retries bounded, and avoid one huge fan-out that can single-handedly blow the whole request's latency budget.